In [1]:
!pip install ultralytics

In [2]:
import os 
import shutil
import random
import pandas as pd
from ultralytics import YOLO
from sklearn.metrics import classification_report

# Split the data into training and validation sets

In [3]:
# konfigurasi path
SOURCE_DIR = 'data/train' 
BASE_OUTPUT_DIR = 'dataset_split'
VAL_RATIO = 0.2

def split_dataset():
    """
    Fungsi ini untuk membagi dataset dari folder train lama menjadi 
    folder train dan val yang baru
    """
    print(f"Memulai pembagian dataset dengan rasio validation: {VAL_RATIO*100}%")
    
    # membuat direktori utama untuk output
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    
    # mendefinisikan path folder train dan val di dalam folder baru
    train_dir = os.path.join(BASE_OUTPUT_DIR, "train")
    val_dir = os.path.join(BASE_OUTPUT_DIR, "val")
    
    # mengambil nama-nama sub-folder kelas (0_Recycable, 1_Electronic, 2_Organic)
    try:
        classes = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR))]
    except FileNotFoundError:
        print(f"Error: Folder '{SOURCE_DIR}' tidak ditemukan. Pastikan path sudah benar.")
        return
    
    # iterasi untuk setiap kelas
    for cls in classes:
        print(f"\nMemproses kelas: {cls}...")
        
        # membuat folder kelas di dalma train dan val yang baru
        os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        
        cls_path = os.path.join(SOURCE_DIR, cls)
        
        # mengambil semua file yang ada di dalam folder kelas tersebut
        images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        
        # mengacak urutan data agar distribusi gambar saat di split merata
        random.seed(42) # menggunakan seed agar hasil acakan tetap konssiten jika skrip diulang
        random.shuffle(images)
        
        # menghitung index pemisah (titik potong array)
        split_idx = int(len(images) * (1-VAL_RATIO))
        
        # membagi list gambar menjadi train dan val
        train_images = images[:split_idx]
        val_images = images[split_idx:]
        
        # proses copy file ke folder train baru
        for img in train_images:
            src = os.path.join(cls_path, img)
            dst = os.path.join(train_dir, cls, img)
            shutil.copy2(src, dst) # copy2 mempertahankan metadata gambar
            
        # proses copy file ke folder val baru
        for img in val_images:
            src = os.path.join(cls_path, img)
            dst = os.path.join(val_dir, cls, img)
            shutil.copy2(src, dst)
            
        print(f"Total gambar: {len(images)}")
        print(f"Masuk Train: {len(train_images)}")
        print(f"Masuk Val: {len(val_images)}")
        
    print(f"\n Proses selesai! Dataset baru yang siap diapakai YOLO ada di folder '{BASE_OUTPUT_DIR}'.")

if __name__ == "__main__":
    split_dataset()

Memulai pembagian dataset dengan rasio validation: 20.0%

Memproses kelas: 0_Recyclable...
Total gambar: 9932
Masuk Train: 7945
Masuk Val: 1987

Memproses kelas: 1_Electronic...
Total gambar: 3855
Masuk Train: 3084
Masuk Val: 771

Memproses kelas: 2_Organic...
Total gambar: 12466
Masuk Train: 9972
Masuk Val: 2494

 Proses selesai! Dataset baru yang siap diapakai YOLO ada di folder 'dataset_split'.


# Train YOLO 

In [ ]:
DATASET_PATH = "dataset_split"

def train_model():
    print("Memulai proses training model YOLO11s-cls....")
    
    model = YOLO("yolo11s-cls.pt")
    
    results = model.train(
        data=DATASET_PATH,
        epochs=100,
        patience=15,
        imgsz=224,
        batch=16,
        workers=0,
        project='trash_classification_model',
        name='yolo11_run1_model',
        
        # Data Augmentation
        degrees=15.0,
        flipud=0.2,
        fliplr=0.5,
        hsv_s=0.2,
        hsv_v=0.2
    )
    
    print("Training Selesai! Model tersimpan di 'trash_classification_1/yolo11_run1_1/weights/best.pt'")

if __name__ == "__main__":
    train_model()

Memulai proses training model YOLO11s-cls....
New https://pypi.org/project/ultralytics/8.4.101 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.98  Python-3.13.1 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_split, degrees=15.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.2, hsv_v=0.2, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train,

KeyboardInterrupt: 

# Evaluation

In [4]:
VAL_DIR = "dataset_split/val"
BEST_MODEL_PATH = "data/Yolo11_small_trash_classification_model.pt"

def evaluate_f1_score():
    print("Memulai evaluasi model menggunakan dataset validation")
    
    if not os.path.exists(BEST_MODEL_PATH):
        print("Error: Model weights belum ada")
        return
    
    model = YOLO(BEST_MODEL_PATH)
    y_true = []
    y_pred = []
    
    classes = [d for d in os.listdir(VAL_DIR) if os.path.isdir(os.path.join(VAL_DIR, d))]
    
    for cls in classes:
        print(f"Mengevaluasi kategori: {cls}...")
        true_class_id = int(cls.split('_')[0])
        
        cls_path = os.path.join(VAL_DIR, cls)
        images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        
        for img_name in images:
            img_path = os.path.join(cls_path, img_name)
            
            result = model(img_path, verbose=False)[0]
            top1_index = result.probs.top1
            raw_class_name = result.names[top1_index]
            
            predicted_class_id = int(raw_class_name.split('_')[0])
            
            y_true.append(true_class_id)
            y_pred.append(predicted_class_id)
            
    target_names = [f"Class {i}" for i in sorted(list(set(y_true)))]
    print("\nDetail Laporan Klasifikasi:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
if __name__ == "__main__":
    evaluate_f1_score()

Memulai evaluasi model menggunakan dataset validation
Mengevaluasi kategori: 0_Recyclable...
Mengevaluasi kategori: 1_Electronic...
Mengevaluasi kategori: 2_Organic...

Detail Laporan Klasifikasi:
              precision    recall  f1-score   support

     Class 0       0.92      0.85      0.89      1987
     Class 1       0.95      0.91      0.93       771
     Class 2       0.89      0.96      0.92      2494

    accuracy                           0.91      5252
   macro avg       0.92      0.91      0.91      5252
weighted avg       0.91      0.91      0.91      5252



# Finetuning

In [6]:
DATASET_PATH = "dataset_split"

def train_model():
    print("Memulai proses ADVANCED FINE-TUNING untuk YOLO11s-cls (Small)...")
    
    # Mulai dari Nano sesuai rencanamu
    model = YOLO("yolo11s-cls.pt")
    
    results = model.train(
        data=DATASET_PATH,
        epochs=100,               
        patience=15,              
        imgsz=224,                
        batch=16,                 
        device=0,                 
        workers=0,                
        
        project="trash_classification_with_yolo11",    
        name="yolo11n_run1", # Ganti nama agar tidak menimpa hasil sebelumnya
        
        # ==========================================
        # ADVANCED FINE-TUNING PARAMETERS
        # ==========================================
        freeze=10,                # Membekukan 10 layer pertama (Backbone YOLO) agar pengetahuan dasarnya tidak hilang
        lr0=0.001,                # Learning rate dikecilkan (defaultnya 0.01) agar update bobotnya lebih halus
        optimizer='AdamW',        # Menggunakan optimizer AdamW yang terkenal sangat stabil untuk fine-tuning model klasifikasi
        weight_decay=0.01,        # Mencegah overfitting dengan memberikan penalti jika model terlalu menghafal
        
        # ==========================================
        # DATA AUGMENTASI (Tetap diaktifkan)
        # ==========================================
        degrees=30.0,            
        flipud=0.5,              
        fliplr=0.5,              
        hsv_s=0.3,               
        hsv_v=0.3,               
        scale=0.2,               
        translate=0.2,           
        erasing=0.2              
    )
    
    # Penyesuaian path penyimpanan otomatis
    original_model_path = "trash_classification_1/yolo11_run1_1/weights/best.pt"
    new_model_name = "model_trash_classification_yolo11_small.pt" 
    
    if os.path.exists(original_model_path):
        shutil.copy2(original_model_path, new_model_name)
        print(f"\n✅ Advanced Fine-Tuning Selesai! Model disimpan sebagai '{new_model_name}'")
    else:
        print("\n❌ Peringatan: File best.pt tidak ditemukan.")

if __name__ == "__main__":
    train_model()

Memulai proses ADVANCED FINE-TUNING untuk YOLO11s-cls (Small)...
New https://pypi.org/project/ultralytics/8.4.102 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.98  Python-3.13.1 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_split, degrees=30.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.3, hsv_v=0.3, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mi

In [7]:
VAL_DIR = "dataset_split/val"
BEST_MODEL_PATH = "data/YOLO11_Small_TrashClassification_Model.pt"

def evaluate_f1_score():
    print("Memulai evaluasi model menggunakan dataset validation")
    
    if not os.path.exists(BEST_MODEL_PATH):
        print("Error: Model weights belum ada")
        return
    
    model = YOLO(BEST_MODEL_PATH)
    y_true = []
    y_pred = []
    
    classes = [d for d in os.listdir(VAL_DIR) if os.path.isdir(os.path.join(VAL_DIR, d))]
    
    for cls in classes:
        print(f"Mengevaluasi kategori: {cls}...")
        true_class_id = int(cls.split('_')[0])
        
        cls_path = os.path.join(VAL_DIR, cls)
        images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        
        for img_name in images:
            img_path = os.path.join(cls_path, img_name)
            
            result = model(img_path, verbose=False)[0]
            top1_index = result.probs.top1
            raw_class_name = result.names[top1_index]
            
            predicted_class_id = int(raw_class_name.split('_')[0])
            
            y_true.append(true_class_id)
            y_pred.append(predicted_class_id)
            
    target_names = [f"Class {i}" for i in sorted(list(set(y_true)))]
    print("\nDetail Laporan Klasifikasi:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
if __name__ == "__main__":
    evaluate_f1_score()

Memulai evaluasi model menggunakan dataset validation
Mengevaluasi kategori: 0_Recyclable...
Mengevaluasi kategori: 1_Electronic...
Mengevaluasi kategori: 2_Organic...

Detail Laporan Klasifikasi:
              precision    recall  f1-score   support

     Class 0       0.95      0.95      0.95      1987
     Class 1       0.98      0.98      0.98       771
     Class 2       0.97      0.96      0.97      2494

    accuracy                           0.96      5252
   macro avg       0.96      0.97      0.97      5252
weighted avg       0.96      0.96      0.96      5252



# Submission

In [3]:
TEST_DIR = "data/test"
SUBMISSION_FILE = "submission.csv"

def generate_submission():
    print("Memulai proses prediksi untuk submission....")
    
    sub_df = pd.read_csv(SUBMISSION_FILE)
    
    best_model_path = "model/YOLO11_Small_TrashClassification_Model.pt"
    if not os.path.exists(best_model_path):
        print("Error: Model wights belum ada.")
        return
    
    model = YOLO(best_model_path)
    predictions = []
    
    for img_id in sub_df['id']:
        img_path = os.path.join(TEST_DIR, f"{img_id}.jpg")
        
        if os.path.exists(img_path):
            result = model(img_path, verbose=False)[0]
            top1_index = result.probs.top1
            raw_class_name = result.names[top1_index]
            
            # Konversi string nama folder ("0_Recycable") menjadi integer (0)
            class_number_str = raw_class_name.split('_')[0]
            predicted_class_id = int(class_number_str)
            
            predictions.append(predicted_class_id)
        else:
            print(f"Peringatan: Gambar {img_id}.jpg tidak tditemukan di folder test!")
            predictions.append(None)
    
    sub_df['predicted'] = predictions
    sub_df.to_csv("submission_Satria Ancol.csv", index=False)
    print("Proses selesai! File 'submission_Satria Ancol.csv' sudah selesai diisi")

if __name__ == "__main__":
    generate_submission()

Memulai proses prediksi untuk submission....
Proses selesai! File 'submission_Satria Ancol.csv' sudah selesai diisi
